# Run the recent methods on Porto data

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Run the cells in order. If a cell fails, copy the error message into the team chat (do not skip ahead).

Status of this notebook: the data conversion and baseline parts were tested; the training cells (DiffMM, HSTGMatch) were written from reading the repositories and have **not** been run by the author. If they do not work, that is expected to be fixable or, in the worst case, you write the report from the literature (see the team guide).

## 0. Check the GPU

In [3]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())

Tue Nov 10 20:00:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Save results to Google Drive

## 2. DiffMM (recent method 1): get the code and install

Colab already has PyTorch. We install the other libraries **without version pins** first (the repo asks for Python 3.11 and torch 2.4.0, which Colab may not have).

In [4]:
!git clone https://github.com/decisionintelligence/DiffMM.git
!pip install -q rtree geopandas networkx einops tqdm

Cloning into 'DiffMM'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 52 (delta 7), reused 52 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 19.52 MiB | 18.25 MiB/s, done.
Resolving deltas: 100% (7/7), done.


## 3. DiffMM quick test (1 epoch)

The first run builds cached data files, which can take a while. If this finishes and prints `test_accuracy`, the setup works. `--batch_size 128` is smaller than the README's 512 to avoid running out of GPU memory on a T4.

In [5]:
!cd DiffMM && python main.py --city porto --keep_ratio 0.1 --epochs 1 --batch_size 128 --gpu_id 0 --train_flag --test_flag

Use GPU: cuda 0
Preparing Data...
edge Num:  11491
node Num:  5330
valid edge Num:  11491
long length:  11653.178220574457 234
width length:  5220.884118697607 105
----- construct cnn graph ------
100% 471764/471764 [00:22<00:00, 20874.41it/s]
traj num: 100% 16000/16000 [00:48<00:00, 328.30it/s]
100% 298269/298269 [00:12<00:00, 23321.31it/s]
traj num: 100% 10000/10000 [00:30<00:00, 330.34it/s]
training dataset shape: 16000
validation dataset shape: 10000
100% 297008/297008 [00:13<00:00, 22010.53it/s]
traj num: 100% 10000/10000 [00:32<00:00, 311.76it/s]
testing dataset shape: 10000
Finish data preparing.
modelModelAllShortCut(
  (encoder): TrajEncoder(
    (road_emb): Sequential(
      (0): Linear(in_features=265, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Norm()
    )
    (point_encoder): PointEncoder(
      (fc_point): Linear(in_features=3, out_features=256, bias=True)
      (transformer): TransformerEncode

### If step 3 fails: pinned environment (Python 3.11, as in the README)

Only run this cell if step 3 fails with import or version errors.

In [6]:
!pip install -q uv
!uv venv --python 3.11 /content/py311
!uv pip install --python /content/py311/bin/python 'numpy<2' torch==2.4.0 rtree==1.0.1 geopandas==0.14.4 networkx==3.3 einops==0.8.0 tqdm
!cd DiffMM && /content/py311/bin/python main.py --city porto --keep_ratio 0.1 --epochs 1 --batch_size 128 --gpu_id 0 --train_flag --test_flag

Using CPython 3.11.18
Creating virtual environment at: py311
Activate with: source py311/bin/activate
Using Python 3.11.18 environment at: py311
Resolved 39 packages in 200ms
Prepared 39 packages in 1m 05s
Installed 39 packages in 396ms
 + attrs==26.1.0
 + certifi==2026.10.23
 + click==8.5.1
 + click-plugins==1.1.1.2
 + cligj==0.7.2
 + einops==0.8.0
 + filelock==4.2.10
 + fiona==1.10.1
 + fsspec==2026.9.0
 + geopandas==0.14.4
 + jinja2==3.1.6
 + markupsafe==3.0.4
 + mpmath==1.3.0
 + networkx==3.3
 + numpy==1.26.4
 + nvidia-cublas-cu12==12.1.3.1
 + nvidia-cuda-cupti-cu12==12.1.105
 + nvidia-cuda-nvrtc-cu12==12.1.105
 + nvidia-cuda-runtime-cu12==12.1.105
 + nvidia-cudnn-cu12==9.1.0.70
 + nvidia-cufft-cu12==11.0.2.54
 + nvidia-curand-cu12==10.3.2.106
 + nvidia-cusolver-cu12==11.4.5.107
 + nvidia-cusparse-cu12==12.1.0.106
 + nvidia-nccl-cu12==2.20.5
 + nvidia-nvjitlink-cu12==12.9.86
 + nvidia-nvtx-cu12==12.1.105
 + packaging==26.3
 + pandas==3.0.6
 + pyproj==3.7.2
 + python-dateutil==4.0.0

## 4. DiffMM real runs

Run **r = 0.1 (about 150 s gap) first**, then **r = 0.025 (about 600 s)**. Do the others only if time allows. 10 epochs is a starting point; the code has early stopping. Note the time each run takes for your report.

In [7]:
%%time
!cd DiffMM && python main.py --city porto --keep_ratio 0.1 --epochs 10 --batch_size 128 --gpu_id 0 --train_flag --test_flag 2>&1 | tee ../diffmm_r0.1.txt

Use GPU: cuda 0
Preparing Data...
edge Num:  11491
node Num:  5330
valid edge Num:  11491
long length:  11653.178220574457 234
width length:  5220.884118697607 105
----- construct cnn graph ------
training dataset shape: 16000
validation dataset shape: 10000
testing dataset shape: 10000
Finish data preparing.
modelModelAllShortCut(
  (encoder): TrajEncoder(
    (road_emb): Sequential(
      (0): Linear(in_features=265, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Norm()
    )
    (point_encoder): PointEncoder(
      (fc_point): Linear(in_features=3, out_features=256, bias=True)
      (transformer): TransformerEncoder(
        (pe): PositionalEncoder()
        (layers): ModuleList(
          (0-1): 2 x EncoderLayer(
            (norm_1): Norm()
            (attn): MultiHeadAttention(
              (q_linear): Linear(in_features=256, out_features=256, bias=True)
              (v_linear): Linear(in_features=256, 

In [8]:
%%time
!cd DiffMM && python main.py --city porto --keep_ratio 0.025 --epochs 10 --batch_size 128 --gpu_id 0 --train_flag --test_flag 2>&1 | tee ../diffmm_r0.025.txt

Use GPU: cuda 0
Preparing Data...
edge Num:  11491
node Num:  5330
valid edge Num:  11491
long length:  11653.178220574457 234
width length:  5220.884118697607 105
----- construct cnn graph ------
traj num: 100%|██████████| 10000/10000 [00:17<00:00, 580.25it/s]
training dataset shape: 16000
validation dataset shape: 10000
traj num: 100%|██████████| 10000/10000 [00:16<00:00, 594.46it/s]
testing dataset shape: 10000
Finish data preparing.
modelModelAllShortCut(
  (encoder): TrajEncoder(
    (road_emb): Sequential(
      (0): Linear(in_features=265, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Norm()
    )
    (point_encoder): PointEncoder(
      (fc_point): Linear(in_features=3, out_features=256, bias=True)
      (transformer): TransformerEncoder(
        (pe): PositionalEncoder()
        (layers): ModuleList(
          (0-1): 2 x EncoderLayer(
            (norm_1): Norm()
            (attn): MultiHeadAttention(

In [9]:
# optional, if time allows
# !cd DiffMM && python main.py --city porto --keep_ratio 0.2 --epochs 10 --batch_size 128 --gpu_id 0 --train_flag --test_flag 2>&1 | tee ../diffmm_r0.2.txt
# !cd DiffMM && python main.py --city porto --keep_ratio 0.05 --epochs 10 --batch_size 128 --gpu_id 0 --train_flag --test_flag 2>&1 | tee ../diffmm_r0.05.txt

### Collect the DiffMM results and save them

In [10]:
!grep -h 'test_accuracy' diffmm_r*.txt
!cp diffmm_r*.txt /content/drive/MyDrive/cm4604_cp3/
!cp -r DiffMM/model /content/drive/MyDrive/cm4604_cp3/diffmm_model

test_accuracy: 0.7420416666666666, test_recall: 0.7425666666666666, test_precision: 0.7422333333333333, test_f1: 0.7423033333333333
test_accuracy: 0.8468109620934622, test_recall: 0.8490669155844156, test_precision: 0.846000725108225, test_f1: 0.8469585752517176
cp: target '/content/drive/MyDrive/cm4604_cp3/': No such file or directory
cp: cannot create directory '/content/drive/MyDrive/cm4604_cp3/diffmm_model': No such file or directory


**Expected:** the DiffMM paper reports about 86% accuracy at the 150 s gap when trained on 16,000 Porto trips (the size of this toy training set). Mid-80s is a healthy result. If you see below 50%, something is wrong; tell the group before writing anything.

## 5. Baselines: HMM and snapping on the same 10,000 test trips

Upload `mm_experiment.py` (from the team guide) to the Colab file panel first.

In [11]:
!pip install -q shapely scipy matplotlib
!python mm_experiment.py --root DiffMM/data/porto --split test --n 10000 --sigma 10 --beta 50 2>&1 | tee baseline_hmm.txt
!cp baseline_hmm.txt results.csv results.png /content/drive/MyDrive/cm4604_cp3/

11491 road segments, 10000 test trajectories

 gap(s)  snap acc%  HMM acc%  snap s/1000  HMM s/1000
     75       69.1      92.1         0.22        5.95
    150       68.7      88.8         0.20        4.54
    300       68.1      83.7         0.11        3.75
    600       67.3      79.0         0.08        3.24
saved results.csv, results.png
cp: target '/content/drive/MyDrive/cm4604_cp3/': No such file or directory


## 6. HSTGMatch (recent method 2): only after DiffMM works

The paper's data are private, so we convert Porto into its input format. Upload `porto_to_hstgmatch.py` to the Colab file panel first.

In [12]:
!git clone https://github.com/Nerooo-g/HSTGMatch.git
!pip install -q torch_geometric pyyaml scikit-learn
!python porto_to_hstgmatch.py --porto DiffMM/data/porto --out HSTGMatch/data/porto_r0.1 --keep_ratio 0.1 --n_train 6000 --n_extra 2000

Cloning into 'HSTGMatch'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 51 (delta 8), reused 40 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 467.20 KiB | 20.31 MiB/s, done.
Resolving deltas: 100% (8/8), done.
wrote 8000 trips, 11491 segments to HSTGMatch/data/porto_r0.1/ (keep_ratio=0.1)


In [13]:
# smoke test that the repo itself runs (uses its own synthetic data)
!cd HSTGMatch && python scripts/smoke_test.py

=== HSTGMatch Smoke Test ===

[1] Testing imports...
    All imports OK

[2] Preprocessing...
    Grids: 529  Segments: 220
    Processed 500 trajectories

[3] Building ATA-Graph...
    Edges: 4696  Weights shape: torch.Size([4696])

[4] GridGraphEncoder forward pass...
    Output shape: torch.Size([529, 32])  (expected [529, 32])

[5] TransformerEncoder forward pass...
    Output shape: torch.Size([2, 10, 32])  (expected [2, 10, 32])

[6] TransformerDecoder forward pass...
    Logits shape: torch.Size([2, 5, 220])  (expected [2, 5, 220])

[7] IntervalEmbedding forward pass...
    Output shape: torch.Size([2, 4, 32])  (expected [2, 4, 32])

[8] HSTGMatchPretrainer forward + loss...
    grid_logits: torch.Size([4, 19, 529])
    coord_preds: torch.Size([4, 19, 2])
    mask: torch.Size([4, 19])  (masked: 21)
    Loss: 3.4282

[9] HSTGMatch full model forward pass...
    Pretrained encoder loaded OK
    Logits shape: torch.Size([4, 12, 220])  (expected [4, *, 220])
    CE Loss: 5.5202
    

In [14]:
%%time
# stage 1: pre-training (short)
!cd HSTGMatch && python scripts/pretrain.py --config configs/default.yaml --data_dir data/porto_r0.1 --epochs 5 --batch_size 32 2>&1 | tee ../hstg_pretrain.txt

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:533: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:177.)
  output = torch._nested_tensor_from_mask(
Using device: cuda
Loading data from data/porto_r0.1...
  Grid size: 54 × 114 = 6156 cells
  Processed 8000 trajectories.
  ATA-Graph: 59528 edges over 6156 nodes
  Train: 7200  Val: 800
Model parameters: 2,474,766

Starting pre-training for 5 epochs...
Epoch    1/5  train_loss=4.4586  val_loss=4.1550  lr=9.05e-05
Epoch    2/5  train_loss=4.0583  val_loss=4.0013  lr=6.55e-05
Epoch    3/5  train_loss=3.9974  val_loss=3.9851  lr=3.45e-05
Epoch    4/5  train_loss=3.9654  val_loss=3.9753  lr=9.55e-06
Epo

In [15]:
%%time
# stage 2: supervised training (short)
!cd HSTGMatch && python scripts/train.py --config configs/default.yaml --data_dir data/porto_r0.1 --pretrained_encoder checkpoints/pretrain/best_encoder.pt --epochs 10 --batch_size 32 2>&1 | tee ../hstg_train.txt

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:533: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:177.)
  output = torch._nested_tensor_from_mask(
Using device: cuda
Loading data from data/porto_r0.1...
  Loading preprocessor state from checkpoints/pretrain/preprocessor.pkl
  Grids: 6156  Segments: 11491
  ATA-Graph edges: 59528
  Split: train=5600  val=800  test=1600
Loading pretrained encoder from checkpoints/pretrain/best_encoder.pt...
  Pretrained encoder loaded successfully.
Model parameters: 5,795,200

Starting supervised training for 10 epochs...
Epoch    1/10  train_loss=8.8863  val_loss=8.4644  P=0.0036  R=0.0036  F1=0.0036  lr=4.88e-0

In [16]:
!cp hstg_*.txt /content/drive/MyDrive/cm4604_cp3/

cp: target '/content/drive/MyDrive/cm4604_cp3/': No such file or directory


## 7. What to write down (copy into the team notes)

- Date, who ran it, runtime type (T4), Python and torch versions
- The exact command for each run, the number of epochs, the time each run took
- Every `test_accuracy` / precision / recall / F1 line, **plus any error message and what you did about it**
- Anything you changed in the code (even one line)